# Pluggable PTC tools

Skein selects one model-visible ADK tool while keeping the host effect broker unchanged.

| Implementation | Tool | Python lifetime | Durable representation |
| --- | --- | --- | --- |
| `skein_notebook` | `python` | Persistent run-scoped heap | Write-ahead ledger events plus a deterministic notebook |
| `adk_code_mode` | `execute_code` | ADK-invocation-scoped Docker sandbox | ADK history; workspace effects still use Skein broker receipts |

The four direct tools remain the default control arm. Neither PTC tool owns authorization or completion.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PtcChoice:
    implementation: str
    tool_name: str
    state_scope: str

CHOICES = {
    "skein_notebook": PtcChoice("skein_notebook", "python", "run"),
    "adk_code_mode": PtcChoice("adk_code_mode", "execute_code", "invocation"),
}

def select_ptc(name):
    return CHOICES[name]

assert select_ptc("skein_notebook").tool_name == "python"
assert select_ptc("adk_code_mode").tool_name == "execute_code"
print(CHOICES)


## One broker, two syntaxes

Both tools reduce nested file and shell requests to the same broker calls. This mocked broker records the operation identity and policy result before returning bounded data.


In [ ]:
class Broker:
    def __init__(self):
        self.receipts = []

    def call(self, operation, arguments, operation_id):
        receipt = {
            "operation_id": operation_id,
            "operation": operation,
            "arguments": arguments,
            "authorization": "allowed",
            "status": "completed",
        }
        self.receipts.append(receipt)
        return {"status": "ok", "data": {"text": "example"}, "receipt": receipt}

broker = Broker()
notebook_result = broker.call("read", {"path": "parser.py"}, "notebook-cell-1:call-1")
code_mode_result = broker.call("read", {"path": "parser.py"}, "code-block-1:call-1")
assert notebook_result["data"] == code_mode_result["data"]
assert {row["operation"] for row in broker.receipts} == {"read"}
assert len({row["operation_id"] for row in broker.receipts}) == 2
print(broker.receipts)


## Different state contracts

Skein notebook PTC can safely reconstruct only committed self-contained data cells. ADK Code Mode discards globals between invocations; durable work must already exist through brokered workspace effects. This difference is part of the ablation, not hidden by an adapter.


In [ ]:
safe_cells = ["target = 'parser.py'", "observations = {'branch': 'union'}"]
notebook_heap = {}
for cell in safe_cells:
    exec(cell, notebook_heap)
restored_heap = {}
for cell in safe_cells:
    exec(cell, restored_heap)
assert restored_heap["observations"] == notebook_heap["observations"]

code_mode_invocation_1 = {"target": "parser.py"}
code_mode_invocation_2 = {}
assert "target" not in code_mode_invocation_2
print({"notebook_restored": True, "code_mode_next_invocation_globals": []})


## Configuration

Use `notebook_ptc.enabled: true` and select `implementation: skein_notebook` or `adk_code_mode`. Code Mode also requires an explicit image tag or digest and the `ptc-adk-code-mode` installation extra. Keep model, task, budgets, and memory strategy fixed when comparing them.
